# AI5707 — Assignment 8
## TI Digits: GMM, LDA, KNN, SVM, PCA and DTW

**Team 16**

This notebook follows the workflow of `02_MNIST.ipynb` and retains the DTW experiment from the original TI Digits notebook. All model implementations and shared utilities are imported by their exact names from `assignment8_common.py`. No model is reimplemented here and no `np.random` function is used.

## 1. Import the exact functions from the common Python file

In [ ]:
import sys
from pathlib import Path
import zipfile

ROOT = Path.cwd()
if not (ROOT / "assignment8_common.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "assignment8_common.py").exists():
    raise FileNotFoundError(
        "assignment8_common.py was not found. Keep it in the project root "
        "alongside the DATASETS folder and this notebook."
    )

sys.path.insert(0, str(ROOT))

# Explicit imports: every name below is defined in assignment8_common.py.
from assignment8_common import (
    np, pd, plt, display,
    accuracy_score, confusion_matrix,
    DATA, OUT, seed,
    deterministic_permutation, deterministic_split,
    standardize_fit, standardize_transform,
    PCAFromScratch, LDAFromScratch, KNNFromScratch,
    GMMFromScratch, gmm_predict, SVMFromScratch,
    evaluate_classifiers, dtw_distance
)

required_functions = [
    deterministic_permutation, deterministic_split,
    standardize_fit, standardize_transform,
    gmm_predict, evaluate_classifiers, dtw_distance
]
required_classes = [
    PCAFromScratch, LDAFromScratch, KNNFromScratch,
    GMMFromScratch, SVMFromScratch
]
assert all(callable(item) for item in required_functions + required_classes)

print("Common file:", ROOT / "assignment8_common.py")
print("Dataset directory:", DATA)
OUT.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUT)
print("All required common functions and model classes imported successfully.")
print("Initial random seed:", seed)

## Models included in this notebook

The TI Digits feature-based pipeline runs **all four classifiers from `assignment8_common.py`**:

1. **GMM** — Gaussian Mixture Model classifier
2. **LDA** — Linear Discriminant Analysis
3. **KNN** — k-Nearest Neighbours (`k=5`)
4. **SVM** — RBF-kernel Support Vector Machine trained with SMO

Each classifier is evaluated on the original standardized features and in the PCA comparison. **DTW** is also evaluated separately on the original variable-length recordings. PCA is a feature transformation, not an additional classifier.


## 2. Extract and load TI Digits data
This follows the original TI Digits notebook: load digits 0–9, deterministically shuffle each digit's recordings, retain up to 40 recordings per class, and split each class into 65% training and 35% testing.

In [ ]:
archive = DATA / "tdigits.zip"
extract_dir = DATA / "tdigits_extracted"

if not archive.exists():
    raise FileNotFoundError(
        f"Dataset archive not found: {archive}. "
        "Place tdigits.zip inside the DATASETS folder."
    )

seq_dir = extract_dir / "Data" / "ExtractedData"
if not seq_dir.exists():
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as zip_file:
        zip_file.extractall(extract_dir)

if not seq_dir.exists():
    raise FileNotFoundError(
        f"Expected extracted digit files under {seq_dir}. "
        "Check the internal folder structure of tdigits.zip."
    )

X_train_seq, y_train_list = [], []
X_test_seq, y_test_list = [], []

for digit in range(10):
    digit_file = seq_dir / f"{digit}.npz"
    if not digit_file.exists():
        print(f"Warning: digit file {digit}.npz is missing.")
        continue

    with np.load(digit_file) as digit_data:
        keys = sorted(
            digit_data.files,
            key=lambda key: int(key.split("_")[-1])
        )
        digit_sequences = [
            np.asarray(digit_data[key], dtype=float)
            for key in keys
        ]

    order = deterministic_permutation(len(digit_sequences))
    selected_sequences = [
        digit_sequences[index]
        for index in order[:min(40, len(order))]
    ]

    split_point = max(1, int(0.65 * len(selected_sequences)))
    train_sequences = selected_sequences[:split_point]
    test_sequences = selected_sequences[split_point:]

    X_train_seq.extend(train_sequences)
    y_train_list.extend([digit] * len(train_sequences))
    X_test_seq.extend(test_sequences)
    y_test_list.extend([digit] * len(test_sequences))

y_train = np.asarray(y_train_list, dtype=int)
y_test = np.asarray(y_test_list, dtype=int)

if len(X_train_seq) == 0 or len(X_test_seq) == 0:
    raise ValueError("No training or testing recordings were loaded.")

print("Training recordings:", len(X_train_seq))
print("Testing recordings:", len(X_test_seq))
print("Training classes/counts:", np.unique(y_train, return_counts=True))
print("Testing classes/counts:", np.unique(y_test, return_counts=True))
print("Frame feature dimension:", X_train_seq[0].shape[1])

## 3. Inspect sequence lengths

In [ ]:
train_lengths = np.asarray([len(sequence) for sequence in X_train_seq])
test_lengths = np.asarray([len(sequence) for sequence in X_test_seq])

plt.figure(figsize=(9, 4))
plt.hist(train_lengths, bins=25, alpha=0.7, label="Training")
plt.hist(test_lengths, bins=25, alpha=0.7, label="Testing")
plt.xlabel("Frames per recording")
plt.ylabel("Number of recordings")
plt.title("TI Digits — sequence length distribution")
plt.legend()
plt.tight_layout()
plt.show()

print("Training sequence length — min:", train_lengths.min())
print("Training sequence length — median:", np.median(train_lengths))
print("Training sequence length — max:", train_lengths.max())

## 4. Convert variable-length sequences into fixed-length vectors

The common classifiers expect a 2D matrix (recordings × features), but TI Digits recordings have different numbers of frames. We uniformly resample each recording to 8 frames and flatten it. This compact representation keeps the temporal order while avoiding unnecessarily large covariance matrices in GMM and LDA. The DTW experiment later uses the original variable-length sequences.

In [ ]:
TARGET_FRAMES = 8
FRAME_FEATURES = X_train_seq[0].shape[1]

def resample_sequence(sequence, target_frames=TARGET_FRAMES):
    sequence = np.asarray(sequence, dtype=float)

    if sequence.ndim != 2 or len(sequence) == 0:
        raise ValueError("Each recording must be a non-empty 2D array.")

    old_time = np.linspace(0.0, 1.0, len(sequence))
    new_time = np.linspace(0.0, 1.0, target_frames)

    output = np.empty((target_frames, sequence.shape[1]), dtype=float)
    for feature in range(sequence.shape[1]):
        output[:, feature] = np.interp(
            new_time, old_time, sequence[:, feature]
        )
    return output

def sequences_to_features(sequence_list):
    resampled = np.asarray([
        resample_sequence(sequence) for sequence in sequence_list
    ])
    return resampled.reshape(len(resampled), -1)

X_train = sequences_to_features(X_train_seq)
X_test = sequences_to_features(X_test_seq)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print(
    f"Each recording: {TARGET_FRAMES} frames × {FRAME_FEATURES} "
    f"features = {X_train.shape[1]} flattened features"
)

## 5. Train-test feature standardization

In [ ]:
# Fit the mean and standard deviation using training data only.
m, s = standardize_fit(X_train)
A = standardize_transform(X_train, m, s)
B = standardize_transform(X_test, m, s)

print("Standardized training data:", A.shape)
print("Standardized testing data:", B.shape)

In [ ]:
# Explicit model roster used throughout the notebook.
MODEL_NAMES = ["GMM", "LDA", "KNN", "SVM"]

print("Classifiers to be evaluated:", ", ".join(MODEL_NAMES))
print("Additional sequence-based method: DTW")


## 6. GMM classifier (EM from common file)

In [ ]:
gmm_pred = gmm_predict(A, y_train, B, K=1)
gmm_accuracy = accuracy_score(y_test, gmm_pred)
print("GMM accuracy:", f"{gmm_accuracy * 100:.2f}%")

## 7. LDA classifier (from common file)

In [ ]:
lda = LDAFromScratch().fit(A, y_train)
lda_pred = lda.predict(B)
lda_accuracy = accuracy_score(y_test, lda_pred)
print("LDA accuracy:", f"{lda_accuracy * 100:.2f}%")

## 8. KNN classifier (from common file)

In [ ]:
knn = KNNFromScratch(k=5).fit(A, y_train)
knn_pred = knn.predict(B)
knn_accuracy = accuracy_score(y_test, knn_pred)
print("KNN accuracy:", f"{knn_accuracy * 100:.2f}%")

## 9. SVM classifier (RBF + SMO from common file)
As in the MNIST notebook, a deterministic subset of at most 1,200 training recordings is used for SVM because its kernel matrix has quadratic memory requirements.

In [ ]:
svm_indices = deterministic_permutation(len(A))[:min(1200, len(A))]
svm = SVMFromScratch(
    C=2, max_passes=4, max_iter=300
).fit(A[svm_indices], y_train[svm_indices])

svm_pred = svm.predict(B)
svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM training recordings:", len(svm_indices))
print("SVM accuracy:", f"{svm_accuracy * 100:.2f}%")

## 10. Compare all classifiers without PCA

In [ ]:
base_predictions = {
    "GMM": gmm_pred,
    "LDA": lda_pred,
    "KNN": knn_pred,
    "SVM": svm_pred
}

# Fail early if any of the four MNIST classifiers has not produced predictions.
assert set(base_predictions) == set(MODEL_NAMES), (
    f"Expected all classifiers {MODEL_NAMES}, got {list(base_predictions)}"
)
assert all(len(prediction) == len(y_test) for prediction in base_predictions.values())

class_labels = np.unique(np.concatenate([y_test] + list(base_predictions.values())))
base_rows = []

for model_name, predictions in base_predictions.items():
    score = accuracy_score(y_test, predictions)
    base_rows.append({
        "Features": "Original",
        "Model": model_name,
        "Accuracy": score
    })

    matrix = confusion_matrix(y_test, predictions, labels=class_labels)
    print(f"\\n{model_name} confusion matrix")
    display(pd.DataFrame(matrix, index=class_labels, columns=class_labels))

base_results_df = pd.DataFrame(base_rows)
display(base_results_df)

## 11. PCA experiments

This section follows the MNIST notebook and calls `evaluate_classifiers()` from the common file. The helper independently standardizes the training data, fits PCA on training data only, transforms the test data, and evaluates GMM, LDA, KNN and SVM. The selected PCA dimensions are 2, 4, 8 and 12, in addition to the original features.

In [ ]:
pca_results = []

for component_count in [None, 2, 4, 8, 12]:
    if component_count is not None and component_count >= A.shape[1]:
        continue

    scores = evaluate_classifiers(
        X_train, y_train, X_test, y_test,
        pca_k=component_count
    )

    row = {
        "Features": (
            "Original" if component_count is None
            else f"PCA-{component_count}"
        ),
        **scores
    }
    pca_results.append(row)
    print(row)

result_df = pd.DataFrame(pca_results)
display(result_df)

## 12. Plot PCA comparison and save results

In [ ]:
ax = result_df.set_index("Features")[MODEL_NAMES].plot(
    kind="bar", ylim=(0, 1), figsize=(10, 5)
)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Feature representation")
ax.set_title("TI Digits — classifiers with and without PCA")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

result_df.to_csv(OUT / "ti_digits_classifier_pca_results.csv", index=False)
print("Saved PCA comparison:", OUT / "ti_digits_classifier_pca_results.csv")

## 13. DTW distance

The DTW distance implementation is imported from `assignment8_common.py` as `dtw_distance`. It is not redefined in this notebook. A small helper below only reduces the number of frames to make pairwise comparisons faster.

In [ ]:
def reduce_sequence(sequence, max_frames=30):
    sequence = np.asarray(sequence, dtype=float)
    if len(sequence) <= max_frames:
        return sequence
    indices = np.linspace(0, len(sequence) - 1, max_frames).astype(int)
    return sequence[indices]

# Quick check that the imported DTW function is callable.
assert callable(dtw_distance)
print("Imported common dtw_distance() successfully.")

## 14. Template-based DTW classification

In [ ]:
templates = {}

for class_label in np.unique(y_train):
    class_recordings = [
        reduce_sequence(sequence)
        for sequence, label in zip(X_train_seq, y_train)
        if label == class_label
    ]
    templates[class_label] = class_recordings[:min(8, len(class_recordings))]

dtw_pred_list = []

for test_sequence in X_test_seq:
    test_sequence = reduce_sequence(test_sequence)
    best_label = None
    best_distance = np.inf

    for class_label, reference_sequences in templates.items():
        distances = [
            dtw_distance(test_sequence, reference)
            for reference in reference_sequences
        ]
        class_distance = min(distances)

        if class_distance < best_distance:
            best_distance = class_distance
            best_label = class_label

    dtw_pred_list.append(best_label)

dtw_pred = np.asarray(dtw_pred_list)
dtw_accuracy = accuracy_score(y_test, dtw_pred)
print("DTW accuracy:", f"{dtw_accuracy * 100:.2f}%")

## 15. DTW confusion matrix

In [ ]:
dtw_cm = confusion_matrix(y_test, dtw_pred, labels=class_labels)
display(pd.DataFrame(dtw_cm, index=class_labels, columns=class_labels))

plt.figure(figsize=(7, 6))
plt.imshow(dtw_cm, aspect="auto")
plt.colorbar()
plt.xticks(range(len(class_labels)), class_labels)
plt.yticks(range(len(class_labels)), class_labels)
plt.xlabel("Predicted digit")
plt.ylabel("True digit")
plt.title("TI Digits — DTW confusion matrix")
plt.tight_layout()
plt.show()

## 16. Final model comparison and prediction export

In [ ]:
final_rows = []

for model_name, predictions in base_predictions.items():
    final_rows.append({
        "Model": model_name,
        "Feature method": "8-frame flattened + standardized",
        "Accuracy": accuracy_score(y_test, predictions)
    })

final_rows.append({
    "Model": "DTW",
    "Feature method": "Original variable-length sequences",
    "Accuracy": dtw_accuracy
})

final_results = pd.DataFrame(final_rows)
display(final_results)

final_results.to_csv(OUT / "ti_digits_all_model_results.csv", index=False)

prediction_table = pd.DataFrame({
    "true_label": y_test,
    "GMM": gmm_pred,
    "LDA": lda_pred,
    "KNN": knn_pred,
    "SVM": svm_pred,
    "DTW": dtw_pred
})
prediction_table.to_csv(
    OUT / "ti_digits_all_model_predictions.csv", index=False
)

print("Saved result files to:", OUT)

## 17. Conclusion

This notebook runs all four classifiers used in the MNIST reference notebook — **GMM, LDA, KNN and SVM** — on the TI Digits dataset. Each classifier produces test predictions and an accuracy score, and all four are compared across the original standardized features and PCA feature representations.

The notebook additionally evaluates **DTW** on the original variable-length recordings. DTW is a separate sequence-based method, not a replacement for any of the four classifiers.

Run the notebook from top to bottom to generate the accuracy tables, confusion matrices, PCA comparison plot, DTW results, and CSV exports. DTW uses the original sequences, while GMM, LDA, KNN and SVM use fixed-length resampled feature vectors; consider this difference when interpreting the comparison.
